In [1]:
import json
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    round as spark_round,
    sum as spark_sum,
    when,
)

# 1. Inisialisasi SparkSession
spark = (
    SparkSession.builder.appName("Tugas6_ETL_Pipeline")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# 2. Generator Data
np.random.seed(33)

# Transaksi (CSV)
n_trx = 5000
kategori_list = [
    "Elektronik",
    "Fashion",
    "Makanan",
    "Rumah Tangga",
    "Kesehatan",
]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(
        pd.date_range("2026-10-01", "2026-10-31"), size=n_trx
    ).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Produk (JSON Lines)
produk = [
    {
        "product_id": i,
        "nama_produk": f"Produk-{i}",
        "kategori": np.random.choice(kategori_list),
        "harga": int(
            np.random.choice([25000, 50000, 75000, 100000, 150000, 250000])
        ),
    }
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
  for p in produk:
    f.write(json.dumps(p) + "\n")

# Ulasan (CSV)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(
        df_trx_tugas6["order_id"], size=n_review, replace=False
    ),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print("Ketiga sumber data berhasil dibuat.")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 21:24:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/30 21:24:06 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Ketiga sumber data berhasil dibuat.


In [2]:
# Membaca data dari masing-masing sumber
df_trx = spark.read.csv(
    "tugas6_transaksi.csv", header=True, inferSchema=True
)
df_produk = spark.read.json("tugas6_produk.json")
df_review = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

# Tampilkan jumlah baris
print("=== JUMLAH BARIS DATA ===")
print("Data Transaksi :", df_trx.count(), "baris")
print("Data Produk    :", df_produk.count(), "baris")
print("Data Ulasan    :", df_review.count(), "baris")

# Tampilkan printSchema()
print("\n=== SKEMA TRANSAKSI ===")
df_trx.printSchema()

print("=== SKEMA PRODUK ===")
df_produk.printSchema()

print("=== SKEMA ULASAN ===")
df_review.printSchema()

=== JUMLAH BARIS DATA ===
Data Transaksi : 5000 baris
Data Produk    : 30 baris
Data Ulasan    : 3500 baris

=== SKEMA TRANSAKSI ===
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

=== SKEMA PRODUK ===
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

=== SKEMA ULASAN ===
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [3]:
# Inner join transaksi dengan produk
df_joined = df_trx.join(df_produk, on="product_id", how="inner")

# Left join transaksi dengan ulasan
df_joined = df_joined.join(df_review, on="order_id", how="left")

# Menambahkan kolom total_pendapatan
df_transformed = df_joined.withColumn(
    "total_pendapatan", col("unit_terjual") * col("harga")
)

df_transformed.select(
    "order_id",
    "product_id",
    "nama_produk",
    "kategori",
    "unit_terjual",
    "harga",
    "total_pendapatan",
    "rating",
).show(5)

+--------+----------+-----------+------------+------------+------+----------------+------+
|order_id|product_id|nama_produk|    kategori|unit_terjual| harga|total_pendapatan|rating|
+--------+----------+-----------+------------+------------+------+----------------+------+
|     TX0|        21|  Produk-21|  Elektronik|           2| 25000|           50000|     1|
|     TX1|         8|   Produk-8|     Fashion|           6|250000|         1500000|  NULL|
|     TX2|        25|  Produk-25|  Elektronik|           4| 25000|          100000|     2|
|     TX3|         3|   Produk-3|     Fashion|           4| 75000|          300000|     5|
|     TX4|        19|  Produk-19|Rumah Tangga|           6| 75000|          450000|  NULL|
+--------+----------+-----------+------------+------------+------+----------------+------+
only showing top 5 rows



In [4]:
# 1. Menambahkan kolom boolean ada_ulasan sebelum na.fill()
df_enriched = df_transformed.withColumn(
    "ada_ulasan", col("rating").isNotNull()
)

# 2. Pengisian nilai kosong (null) pada kolom rating dengan angka 0
df_enriched = df_enriched.na.fill({"rating": 0})

df_enriched.select(
    "order_id",
    "nama_produk",
    "kategori",
    "rating",
    "ada_ulasan",
    "total_pendapatan",
).show(10)

+--------+-----------+------------+------+----------+----------------+
|order_id|nama_produk|    kategori|rating|ada_ulasan|total_pendapatan|
+--------+-----------+------------+------+----------+----------------+
|     TX0|  Produk-21|  Elektronik|     1|      true|           50000|
|     TX1|   Produk-8|     Fashion|     0|     false|         1500000|
|     TX2|  Produk-25|  Elektronik|     2|      true|          100000|
|     TX3|   Produk-3|     Fashion|     5|      true|          300000|
|     TX4|  Produk-19|Rumah Tangga|     0|     false|          450000|
|     TX5|  Produk-10|  Elektronik|     5|      true|          225000|
|     TX6|  Produk-26|   Kesehatan|     4|      true|          450000|
|     TX7|   Produk-4|     Makanan|     0|     false|          150000|
|     TX8|   Produk-7|  Elektronik|     5|      true|          150000|
|     TX9|  Produk-30|  Elektronik|     0|     false|          250000|
+--------+-----------+------------+------+----------+----------------+
only s

> **Alasan Pengisian Nilai 0 pada Rating:**  
> Angka `0` digunakan sebagai penanda (*flag*) khusus yang menunjukkan kondisi "belum/tidak ada ulasan", mengingat skala ulasan resmi berkisar antara 1 hingga 5. Mengisi data kosong menggunakan nilai rata-rata (*mean*) atau median akan membiaskan statistik kepuasan pelanggan secara tidak nyata. Dengan menyimpannya sebagai `0` bersama kolom `ada_ulasan`, analisis agregasi rating di masa mendatang dapat menyaring transaksi tanpa ulasan secara akurat (`WHERE rating > 0` atau `WHERE ada_ulasan = True`).

In [5]:
# Membuat direktori HDFS target
!hdfs dfs -mkdir -p /user/zeno/tugas6/hasil_etl

# Menuliskan ke HDFS format Parquet terpartisi berdasarkan kategori
hdfs_path = "hdfs://localhost:9000/user/zeno/tugas6/hasil_etl"
df_enriched.write.mode("overwrite").partitionBy("kategori").parquet(hdfs_path)

print("Data berhasil disimpan ke HDFS dalam format Parquet terpartisi.\n")

# Verifikasi struktur direktori HDFS
!hdfs dfs -ls -R /user/zeno/tugas6/hasil_etl

[Stage 22:>                                                         (0 + 1) / 1]

Data berhasil disimpan ke HDFS dalam format Parquet terpartisi.



-rw-r--r--   3 zeno supergroup          0 2026-09-30 21:31 /user/zeno/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - zeno supergroup          0 2026-09-30 21:31 /user/zeno/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 zeno supergroup      16364 2026-09-30 21:31 /user/zeno/tugas6/hasil_etl/kategori=Elektronik/part-00000-e25727d7-c1a1-4c77-bf10-b760e8905a1f.c000.snappy.parquet
drwxr-xr-x   - zeno supergroup          0 2026-09-30 21:31 /user/zeno/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 zeno supergroup      11134 2026-09-30 21:31 /user/zeno/tugas6/hasil_etl/kategori=Fashion/part-00000-e25727d7-c1a1-4c77-bf10-b760e8905a1f.c000.snappy.parquet
drwxr-xr-x   - zeno supergroup          0 2026-09-30 21:31 /user/zeno/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 zeno supergroup      10393 2026-09-30 21:31 /user/zeno/tugas6/hasil_etl/kategori=Kesehatan/part-00000-e25727d7-c1a1-4c77-bf10-b760e8905a1f.c000.snappy.parquet
drwxr-xr-x   - zeno supergroup          0 2026-09-30 21:31 /user/z

In [6]:
# Membaca kembali data Parquet dari HDFS untuk verifikasi integritas
df_hdfs_check = spark.read.parquet(hdfs_path)

print(f"Jumlah baris terverifikasi dari HDFS: {df_hdfs_check.count()} baris")
df_hdfs_check.printSchema()

Jumlah baris terverifikasi dari HDFS: 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- harga: long (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- total_pendapatan: long (nullable = true)
 |-- ada_ulasan: boolean (nullable = true)
 |-- kategori: string (nullable = true)



In [7]:
# Menghitung persentase ulasan per kategori
df_insight = (
    df_enriched.groupBy("kategori")
    .agg(
        count("order_id").alias("total_transaksi"),
        spark_sum(when(col("ada_ulasan") == True, 1).otherwise(0)).alias(
            "transaksi_berulasan"
        ),
    )
    .withColumn(
        "persentase_berulasan",
        spark_round(
            (col("transaksi_berulasan") / col("total_transaksi")) * 100, 2
        ),
    )
    .orderBy("persentase_berulasan")
)

df_insight.show()

+------------+---------------+-------------------+--------------------+
|    kategori|total_transaksi|transaksi_berulasan|persentase_berulasan|
+------------+---------------+-------------------+--------------------+
|     Makanan|            536|                369|               68.84|
|   Kesehatan|            961|                662|               68.89|
|     Fashion|           1035|                726|               70.14|
|Rumah Tangga|            815|                575|               70.55|
|  Elektronik|           1653|               1168|               70.66|
+------------+---------------+-------------------+--------------------+



### Interpretasi Bisnis
Kategori produk dengan persentase ulasan paling rendah menunjukkan potensi rendahnya keterikatan pelanggan (*customer engagement*) atau kurangnya dorongan setelah proses pembelian (*post-purchase engagement*). Informasi ini penting bagi tim *marketing* untuk merancang strategi tindak lanjut, seperti memberikan insentif berupa poin loyalitas atau kupon diskon khusus bagi pembeli pada kategori tersebut agar bersedia memberikan ulasan. Selain itu, tim dapat mengevaluasi apakah produk pada kategori tersebut mengalami kendala pengalaman pengguna atau kualitas yang membuat pelanggan enggan berinteraksi lebih jauh.